RQs
1. What predicts data viz literacy multiple choice item difficulty most between visual features (i.e. data viz image) vs text features (i.e. question text and the possible responses)?
2. What about the combination between the visual and text features?

In [ ]:
import pandas as pd
import numpy as np
from openai import OpenAI
import dotenv

dotenv.load_dotenv()
client = OpenAI()

# Load Dataset

In [ ]:
df = pd.read_csv("psych_139_mini_project_1_split_80_responses.csv", index_col=0)
# Transform df such that we get item_id aggregate by average of correct_response + relevant covariates 
df['incorrect_response'] = df.correct_response.apply(lambda x: 1 if x == 0 else 0)
df = df.groupby(['item_id', 'image_url', 'graph_type', 'test_name', 'task_category', 'possible_responses', 'question_text'])['incorrect_response'].mean().reset_index()
df = df.rename(columns={'incorrect_response': 'difficulty'})

# create easiness score 
df['easiness'] = 1 - df['difficulty']
df.head()


# Create Features that LLM Describes Prior to Prediction + System Prompt

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional

# ==================================================
# Pydantic Model 1: Visualization Factors (Visual Only)
# ==================================================
class VisualizationFactors(BaseModel):
    """Analyzes factors related to the data visualization itself."""
    readability: str = Field(..., description="Assessment of text legibility, contrast, and overall visual ease of reading.")
    annotations_clarity: str = Field(..., description="Description of any annotations (e.g., titles, legends, average lines) and their potential impact on interpretation.")
    axis_clarity: str = Field(..., description="Description of axis clarity (labels, scale, intervals, range).")
    data_encoding_clarity: str = Field(..., description="Description of how clearly data is represented and mappable to axes.")
    visual_clutter_level: str = Field(..., description="Level of visual clutter or 'chart junk' (e.g., 'Low', 'Medium', 'High').")
    data_series_count: int = Field(..., description="Number of distinct data series plotted.")
    chart_type: str = Field(..., description="Type of chart used (e.g., 'Line Graph', 'Bar Chart', 'Scatter Plot').")

    overall_visual_complexity: str = Field(..., description="Subjective assessment of the visualization's overall complexity (e.g., 'Low', 'Medium', 'High').")
    
    prediction: float = Field(
        ..., 
        description="Estimated probability (0-1) a random US adult answers correctly, considering *primarily* the strengths and weaknesses of the visualization itself."
    )

SYSTEM_PROMPT_VISION = f"""
You are an expert in data visualization literacy. Your task is to analyze the intrinsic visual characteristics of a data visualization presented in an image. 
Focus *exclusively* on the factors related to the visualization itself (chart type, clarity, complexity, encoding, readability, annotations, clutter, etc.). 
Do *not* consider any specific question or multiple-choice options that might be associated with this visualization elsewhere. Your analysis should be based solely on the visual presentation.

Input:
- image_url: URL pointing to the visualization image.

Output Instructions:
Analyze the visualization based purely on the visual elements in the image at the provided `image_url`. Provide your analysis as a JSON object that strictly conforms to the following Pydantic model structure:

```json
{{
  "chart_type": "Type of chart used (e.g., 'Line Graph', 'Bar Chart', 'Scatter Plot').",
  "axis_clarity": "Description of axis clarity (labels, scale, intervals, range).",
  "data_encoding_clarity": "Description of how clearly data is represented and mappable to axes.",
  "readability": "Assessment of text legibility, contrast, and overall visual ease of reading.",
  "visual_clutter_level": "Level of visual clutter or 'chart junk' (e.g., 'Low', 'Medium', 'High').",
  "data_series_count": "Number of distinct data series plotted.",
  "annotations_clarity": "Description of any annotations (e.g., titles, legends, average lines) and their potential impact on interpretation.",
  "overall_visual_complexity": "Subjective assessment of the visualization's overall complexity (e.g., 'Low', 'Medium', 'High').",
  "prediction": "Estimated probability (0.0 to 1.0) a random US adult could accurately extract information from this visualization IF asked a simple, clear question, based *only* on the visualization's strengths/weaknesses."
}}"""

# ==================================================
# Pydantic Model 2: Combined Question & Answer Factors (Text Only)
# ==================================================
class QuestionAndAnswerFactorsTextOnly(BaseModel):
    """
    Analyzes factors related to the question and answer options, 
    based solely on text input without visual context.
    """
    # Question Factors
    cognitive_task_type: str = Field(..., description="Type of cognitive task implied by the question (e.g., 'Value Lookup', 'Comparison', 'Calculation', 'Identification').")
    question_clarity: str = Field(..., description="Assessment of the question's wording, ambiguity, and directness based on text alone.")
    information_integration_level: str = Field(..., description="Level of information integration implied by the question structure (e.g., 'Low - single concept', 'Medium - requires relating parts of question').")
    
    # Answer Option Factors
    num_options: int = Field(..., description="The total number of multiple-choice options.")
    correct_answer: str = Field(..., description="The text of the correct answer option.")
    distractor_analysis: str = Field(..., description="Analysis of incorrect options (distractors) based on text: plausibility, numerical proximity, format, potential conceptual errors (cannot reference visual elements).")
    format_consistency: str = Field(..., description="Whether all options share a consistent format (e.g., units, data type) based on text and why.")
    
    # Prediction
    prediction: float = Field(
        ...,
        description="Estimated probability (0-1) a random US adult answers correctly, considering *only* the textual factors of the question and options, independent of visual interpretation difficulty."
    )

# ==================================================
# System Prompt: Analyze Question & Answer Factors (Text Only)
# ==================================================
SYSTEM_PROMPT_TEXT = f"""
You are an expert in psychometrics, cognitive science, and test design. Your task is to analyze the textual characteristics of a question and its corresponding multiple-choice answer options. 
Focus *exclusively* on the factors related to the question text and the answer option texts. 
You will *not* receive the visualization image; therefore, your analysis cannot depend on visual elements or specific values read from a chart.

Input:
- question_text: The specific question being asked.
- options_list: The list of multiple-choice options as strings.

Output Instructions:
Analyze the question and answer options based purely on the provided text. Provide your analysis as a JSON object that strictly conforms to the following Pydantic model structure:

```json
{{
  "cognitive_task_type": "Type of cognitive task implied by the question (e.g., 'Value Lookup', 'Comparison', 'Calculation', 'Identification').",
  "question_clarity": "Assessment of the question's wording, ambiguity, and directness based on text alone.",
  "information_integration_level": "Level of information integration implied by the question structure (e.g., 'Low - single concept', 'Medium - requires relating parts of question').",
  "num_options": "The total number of multiple-choice options.",
  "correct_answer": "The text of the correct answer option.",
  "distractor_analysis": "Analysis of incorrect options (distractors) based on text: plausibility, numerical proximity, format, potential conceptual errors (cannot reference visual elements).",
  "format_consistency": "Boolean: true if all options share a consistent format (e.g., units, data type) based on text.",
  "prediction": "Estimated probability (0.0 to 1.0) a random US adult answers correctly, considering *only* the textual factors of the question and options, independent of visual interpretation difficulty."
}}"""
# ==================================================
# Pydantic Model: Comprehensive Factors (Vision + Text)
# ==================================================
class VisionAndTextFactors(BaseModel):
    """
    Analyzes factors related to the visualization, question, and answer options 
    using both visual and textual input for a holistic difficulty assessment.
    """
    # Visualization Factors Summary
    chart_type: str = Field(..., description="Type of chart used (e.g., 'Line Graph', 'Bar Chart', 'Scatter Plot').")
    axis_clarity: str = Field(..., description="Description of axis clarity (labels, scale, intervals, range).")
    data_encoding_clarity: str = Field(..., description="Description of how clearly data is represented and mappable to axes.")
    readability: str = Field(..., description="Assessment of text legibility, contrast, and overall visual ease of reading.")
    visual_clutter_level: str = Field(..., description="Level of visual clutter or 'chart junk' (e.g., 'Low', 'Medium', 'High').")
    data_series_count: int = Field(..., description="Number of distinct data series plotted.")
    annotations_clarity: str = Field(..., description="Description of any annotations (e.g., titles, legends, average lines) and their potential impact on interpretation.")
    overall_visual_complexity: str = Field(..., description="Subjective assessment of the visualization's overall complexity (e.g., 'Low', 'Medium', 'High').")
    
    # Question Factors
    cognitive_task_type: str = Field(..., description="Type of cognitive task implied by the question (e.g., 'Value Lookup', 'Comparison', 'Calculation', 'Identification').")
    question_clarity: str = Field(..., description="Assessment of the question's wording, ambiguity, and directness based on text alone.")
    information_integration_level: str = Field(..., description="Level of information integration implied by the question structure (e.g., 'Low - single concept', 'Medium - requires relating parts of question').")
    
    # Answer Option Factors
    num_options: int = Field(..., description="The total number of multiple-choice options.")
    correct_answer: str = Field(..., description="The text of the correct answer option.")
    distractor_analysis: str = Field(..., description="Analysis of incorrect options (distractors) based on text: plausibility, numerical proximity, format, potential conceptual errors (cannot reference visual elements).")
    format_consistency: str = Field(..., description="Whether all options share a consistent format (e.g., units, data type) based on text and why.")
    
    # Interaction & Overall Assessment
    visual_text_alignment: str = Field(..., description="Assessment of how well the terms in the question and options map to labels and elements in the visualization.")
    required_precision: str = Field(..., description="Precision needed to extract the correct value (e.g., 'Exact gridline', 'Interpolation required', 'Categorical').")
    overall_difficulty_assessment: str = Field(..., description="Qualitative summary explaining the interplay of factors and justifying the predicted difficulty.")
    
    # Prediction
    prediction: float = Field(
        ..., 
        description="Final estimated probability (0-1) a random US adult answers this specific question correctly, integrating all visual, textual, and interaction factors."
    )

# ==================================================
# System Prompt: Comprehensive Analysis (Vision + Text)
# ==================================================
SYSTEM_PROMPT_VISION_AND_TEXT = f"""
You are an expert system synthesizing insights from data visualization literacy, cognitive science, and psychometrics. Your task is to perform a comprehensive analysis of a data visualization question item, considering the interplay between the visualization, the question, and the answer options to predict its overall difficulty for a random US adult.

Input:
- image_url: URL pointing to the visualization image.
- question_text: The specific question being asked about the visualization.
- options_list: The list of multiple-choice options as strings.

Output Instructions:
Analyze all provided inputs holistically. Evaluate the visual elements, the question's demands, the quality of the answer options (especially distractors in visual context), and how these factors interact. Provide your complete analysis as a JSON object that strictly conforms to the following Pydantic model structure:

```json
{{
  "chart_type": "Type of chart used (e.g., 'Line Graph', 'Bar Chart', 'Scatter Plot').",
  "axis_clarity": "Description of axis clarity (labels, scale, intervals, range).",
  "data_encoding_clarity": "Description of how clearly data is represented and mappable to axes.",
  "readability": "Assessment of text legibility, contrast, and overall visual ease of reading.",
  "visual_clutter_level": "Level of visual clutter or 'chart junk' (e.g., 'Low', 'Medium', 'High').",
  "data_series_count": "Number of distinct data series plotted.",
  "annotations_clarity": "Description of any annotations (e.g., titles, legends, average lines) and their potential impact on interpretation.",
  "overall_visual_complexity": "Subjective assessment of the visualization's overall complexity (e.g., 'Low', 'Medium', 'High').",
  "cognitive_task_type": "Type of cognitive task implied by the question (e.g., 'Value Lookup', 'Comparison', 'Calculation', 'Identification').",
  "question_clarity": "Assessment of the question's wording, ambiguity, and directness based on text alone.",
  "information_integration_level": "Level of information integration implied by the question structure (e.g., 'Low - single concept', 'Medium - requires relating parts of question').",
  "num_options": "The total number of multiple-choice options.",
  "correct_answer": "The text of the correct answer option.",
  "distractor_analysis": "Analysis of incorrect options (distractors) based on text: plausibility, numerical proximity, format, potential conceptual errors (cannot reference visual elements).",
  "format_consistency": "Whether all options share a consistent format (e.g., units, data type) based on text and why.",
  "visual_text_alignment": "Assessment of how well the terms in the question and options map to labels and elements in the visualization.",
  "required_precision": "Precision needed to extract the correct value (e.g., 'Exact gridline', 'Interpolation required', 'Categorical').",
  "overall_difficulty_assessment": "Qualitative summary explaining the interplay of factors and justifying the predicted difficulty.",
  "prediction": "Final estimated probability (0.0 to 1.0) a random US adult answers this specific question correctly, integrating all visual, textual, and interaction factors."
}}"""

# LLM API Calls

In [ ]:
def vision_prediction(image_url, model = "gpt-4.1-nano"):
    # get gpt 4.1 mini to explain the image 
    response = client.beta.chat.completions.parse(
        model=model,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": SYSTEM_PROMPT_VISION},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_url,
                    },
                },
            ],
        }],
        response_format=VisualizationFactors,
    )
    return eval(response.choices[0].message.content)["prediction"]

def text_prediction(question_text, options_list, model = "gpt-4.1-nano"):
    response = client.beta.chat.completions.parse(
        model=model,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": SYSTEM_PROMPT_TEXT},
                {"type": "text", "text": f"This is the question:\n{question_text}\nThis is the answer options list:\n{options_list}"},
            ],
        }],
        response_format=QuestionAndAnswerFactorsTextOnly,
    )
    return eval(response.choices[0].message.content)["prediction"]

def vision_and_text_prediction(image_url, question_text, options_list, model = "gpt-4.1-nano"):
    response = client.beta.chat.completions.parse(
        model=model,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": SYSTEM_PROMPT_VISION_AND_TEXT},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_url,
                    },
                },
                {"type": "text", "text": f"This is the question:\n{question_text}\nThis is the answer options list:\n{options_list}"},
            ],
        }],
        response_format=VisionAndTextFactors,
    )
    return response.choices[0].message.content
            

In [ ]:
# Test each on example
example_image_path = "https://data-visualization-benchmark.s3.us-west-2.amazonaws.com/wainer/images/line.png"
example_question = "How much does it rain in March?"
example_options = "['30 mm', '40 mm', '50 mm ', '60 mm']"
samins_guess_probability = 0.5

print("Samin's Guess Probability of Correct Response:")
print(samins_guess_probability)
print("Vision Prediction:")
print(vision_prediction(example_image_path))
print("Text Prediction:")
print(text_prediction(example_question, example_options))
print("Vision + Text Prediction:")
print(vision_and_text_prediction(example_image_path, example_question, example_options))

# Evaluate performance of 3 models 

In [ ]:
# replicate on all held out test
print("Total num of images in set")
print(df.shape)
print("Number of not .png images in test set:")
print(df[~df.image_url.str.endswith('.png')].shape)

count = 0
df_png = df[df.image_url.str.endswith('.png')]
print("Total number of PNG images in test set:")
print(df_png.shape)
for i, row in df_png.iterrows():
    print(count)
    count += 1
    vision_prediction_output = vision_prediction(row.image_url)
    text_prediction_output = text_prediction(row.question_text, row.possible_responses)
    vision_and_text_prediction_output = eval(vision_and_text_prediction(row.image_url, row.question_text, row.possible_responses))["prediction"]
    print(vision_prediction_output)
    print(text_prediction_output)
    print(vision_and_text_prediction_output)

    # Save to df
    df_png.loc[i, 'vision_prediction'] = vision_prediction_output
    df_png.loc[i, 'text_prediction'] = text_prediction_output
    df_png.loc[i, 'proportion_correct'] = vision_and_text_prediction_output
    df_png.loc[i, 'prediction_explanation'] = vision_and_text_prediction_output_raw

In [ ]:
df_png.to_csv("Datasci 194L Mini Project 1/validation_predictions.csv", index=False)

In [ ]:
# calculate absolute difference for each model
absolute_difference_vision = np.abs(df_png.easiness - df_png.vision_prediction)
absolute_difference_text = np.abs(df_png.easiness - df_png.text_prediction)
absolute_difference_vision_and_text = np.abs(df_png.easiness - df_png.proportion_correct)

# Get MAE for each model
mae_vision = absolute_difference_vision.mean()
mae_text = absolute_difference_text.mean()
mae_vision_and_text = absolute_difference_vision_and_text.mean()
    
print(mae_vision)
print(mae_text)
print(mae_vision_and_text)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Model names
models = ['Text', 'Vision', 'Vision + Text']

# Mean Absolute Errors (already computed)
mae_values = [mae_text, mae_vision, mae_vision_and_text]

# Number of samples (assumed the same for all, adjust if needed)
n_samples = len(absolute_difference_vision)

# Calculate standard errors (standard deviation divided by sqrt(n))
sem_text = np.std(absolute_difference_text, ddof=1) / np.sqrt(n_samples)
sem_vision = np.std(absolute_difference_vision, ddof=1) / np.sqrt(n_samples)
sem_vision_and_text = np.std(absolute_difference_vision_and_text, ddof=1) / np.sqrt(n_samples)
sem_values = [sem_text, sem_vision, sem_vision_and_text]

# Create a bar plot with error bars
plt.figure(figsize=(10, 6))
plt.bar(models, mae_values, color=['#90EE90', '#ADD8E6', '#FFB6C1'], yerr=sem_values, capsize=5)
plt.xlabel('Models')
plt.ylabel('Mean Absolute Error (MAE)')
plt.title('Model Performance in Predicting Data Visualization Literacy Item Difficulty')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# plot distribution of predictions for each model
plt.figure(figsize=(10, 6))
plt.hist(df_png.vision_prediction, bins=20, color='blue', alpha=0.5, label='Vision')
plt.hist(df_png.text_prediction, bins=20, color='green', alpha=0.5, label='Text')
plt.hist(df_png.proportion_correct, bins=20, color='red', alpha=0.5, label='Vision + Text')
plt.xlabel('Predicted Easiness Score')
plt.ylabel('Frequency')
plt.title('Distribution of Model Predictions')
plt.legend()
plt.show()


# Attempt best model on test set

In [ ]:
df_test = pd.read_csv("Datasci 194L Mini Project 1/test_data.csv")

print("Number of not .png images in test set:")
print(df_test[~df_test.image_url.str.endswith('.png')].shape)
df_test["prediction_explanation"] = ""

count = 0
print("Total number of images in test set:")
print(df_test.shape)
for i, row in df_test.iterrows():
    print(count)
    count += 1
    if not row.image_url.endswith('.png'):
        continue
    vision_prediction_output = vision_prediction(row.image_url)
    # text_prediction_output = text_prediction(row.question_text, row.possible_responses)
    vision_and_text_prediction_output_raw = vision_and_text_prediction(row.image_url, row.question_text, row.possible_responses)
    vision_and_text_prediction_output = eval(vision_and_text_prediction_output_raw)["prediction"]
    # print(vision_prediction_output)
    # print(text_prediction_output)
    print(vision_and_text_prediction_output)

    # Save to df
    # df_test.loc[i, 'vision_prediction'] = vision_prediction_output
    # df_test.loc[i, 'text_prediction'] = text_prediction_output
    df_test.loc[i, 'proportion_correct'] = vision_and_text_prediction_output
    df_test.loc[i, 'prediction_explanation'] = vision_and_text_prediction_output_raw


In [ ]:
# output csv of test_predictions
df_test[['item_id', 'proportion_correct']].to_csv("Datasci 194L Mini Project 1/test_predictions_all_features.csv", index=False)